## SETUP

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import sys
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())
    print("GPU:", torch.cuda.get_device_name(0))
    print("Capability:", torch.cuda.get_device_capability(0))

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch: 2.10.0+cu128
CUDA: 12.8
CUDA available: True
GPU count: 1
GPU: Tesla T4
Capability: (7, 5)


In [2]:
# %pip install -r /kaggle/input/datasets/hao5234/nlpasses/requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [3]:
import re
import json
import time
import random
import platform
from functools import partial

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset

import transformers
import peft
import accelerate
import bitsandbytes as bnb

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    set_seed,
)
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from rouge_score import rouge_scorer
import textstat

PROCESSED_DIR = "/kaggle/input/datasets/hao5234/nlpasses"
TRAIN_PATH = os.path.join(PROCESSED_DIR, "train.jsonl")
VAL_PATH = os.path.join(PROCESSED_DIR, "validation.jsonl")
TEST_PATH = os.path.join(PROCESSED_DIR, "test.jsonl")
TEMPLATES_PATH = os.path.join(PROCESSED_DIR, "prompt_templates.json")

MODEL_NAME = "facebook/opt-125m"
FINETUNED_MODEL_DIR = "/kaggle/working/models/opt-125m-news-simplifier"
RESULTS_DIR = "/kaggle/working/results"
RESULT_PREFIX = "opt125m"

SEED = 42
MAX_CONTEXT = 2048
MAX_TARGET_TOKENS = 384
MAX_NEW_TOKENS = 384

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

NUM_TRAIN_EPOCHS = 10
EARLY_STOPPING_PATIENCE = 4
EVAL_STEPS = 25
SAVE_STEPS = 25

TRAIN_BATCH_SIZE = 1
EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.05
MAX_GRAD_NORM = 0.3

os.makedirs(FINETUNED_MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA-capable GPU is required for QLoRA training.")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

GPU_CAPABILITY = torch.cuda.get_device_capability(0)
USE_BF16 = GPU_CAPABILITY[0] >= 8 and torch.cuda.is_bf16_supported()
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)
print("bitsandbytes:", bnb.__version__)
print("Compute dtype:", COMPUTE_DTYPE)

Transformers: 4.57.6
PEFT: 0.19.1
Accelerate: 1.13.0
bitsandbytes: 0.50.1
Compute dtype: torch.float16


## Load data and prompt template

In [4]:
REQUIRED_TOP_LEVEL_FIELDS = {"article_id", "article", "summary", "aspects"}
ASPECT_KEYS = ("event", "people", "organization", "date_time", "location", "causes", "effects", "countermeasures")

def load_jsonl(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Could not find '{path}'.")
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            record = json.loads(line)
            missing_top = REQUIRED_TOP_LEVEL_FIELDS - set(record)
            if missing_top:
                raise ValueError(f"{path}, line {line_number}: missing fields {sorted(missing_top)}")
            if not isinstance(record["aspects"], dict):
                raise ValueError(f"{path}, line {line_number}: 'aspects' must be an object.")
            missing_aspects = [key for key in ASPECT_KEYS if key not in record["aspects"]]
            if missing_aspects:
                raise ValueError(f"{path}, line {line_number}: missing aspect fields {missing_aspects}")
            records.append(record)
    if not records:
        raise ValueError(f"No records found in '{path}'.")
    return pd.DataFrame(records)

train_df = load_jsonl(TRAIN_PATH)
val_df = load_jsonl(VAL_PATH)
test_df = load_jsonl(TEST_PATH)

if not os.path.exists(TEMPLATES_PATH):
    raise FileNotFoundError(f"Could not find '{TEMPLATES_PATH}'.")

with open(TEMPLATES_PATH, "r", encoding="utf-8") as f:
    prompt_templates = json.load(f)

PROMPT_TEMPLATE = prompt_templates["prompt_template"]
if PROMPT_TEMPLATE.count("{article}") != 1:
    raise ValueError("prompt_template must contain {article} exactly once.")

print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))
print("\nPrompt template:\n")
print(PROMPT_TEMPLATE)

Training: 400
Validation: 50
Test: 50

Prompt template:

Simplify the news article using only facts stated in the article.
Use simpler vocabulary, grammar, and sentence structures while preserving the meaning.
Do not guess information.

Output exactly this format:

Summary:
<2 to 4 concise simplified sentences>

Aspects:
Event: <main event>
People: <relevant named people only, or Not stated>
Organization: <relevant named organizations only, or Not stated>
Date & Time: <absolute dates/times only, or Not stated>
Location: <relevant locations, or Not stated>
Causes: <cause of the event, or Not stated; use "Believed cause:" if uncertain>
Effects: <results or consequences of the event, or Not stated>
Countermeasures: <actions taken in response to the event, or Not stated>

Use every label exactly once and in the same order.
Do not repeat the same information across Causes, Effects, and Countermeasures.
For Date & Time, do not use relative expressions such as today, yesterday, or last week.


## Build target and fit articles to context

In [5]:
def build_prompt(article: str) -> str:
    return PROMPT_TEMPLATE.format(article=str(article).strip())

def build_training_target(summary: str, aspects: dict) -> str:
    return (
        f"Summary:\n{str(summary).strip()}\n\n"
        "Aspects:\n"
        f"Event: {str(aspects['event']).strip()}\n"
        f"People: {str(aspects['people']).strip()}\n"
        f"Organization: {str(aspects['organization']).strip()}\n"
        f"Date & Time: {str(aspects['date_time']).strip()}\n"
        f"Location: {str(aspects['location']).strip()}\n"
        f"Causes: {str(aspects['causes']).strip()}\n"
        f"Effects: {str(aspects['effects']).strip()}\n"
        f"Countermeasures: {str(aspects['countermeasures']).strip()}"
    )

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
tokenizer.model_max_length = MAX_CONTEXT

def token_length(text: str, add_special_tokens: bool = True) -> int:
    return len(tokenizer(str(text), add_special_tokens=add_special_tokens, truncation=False)["input_ids"])

def fit_prompt_to_context(article: str, reserved_tokens: int):
    input_limit = MAX_CONTEXT - reserved_tokens
    if input_limit <= 0:
        raise ValueError("reserved_tokens must be smaller than MAX_CONTEXT.")

    article = str(article).strip()
    article_ids = tokenizer(article, add_special_tokens=False, truncation=False)["input_ids"]
    empty_prompt_tokens = token_length(build_prompt(""), add_special_tokens=True)
    article_budget = max(0, input_limit - empty_prompt_tokens - 8)
    used_ids = article_ids[:article_budget]

    while True:
        used_article = tokenizer.decode(used_ids, skip_special_tokens=True).strip()
        prompt = build_prompt(used_article)
        prompt_tokens = token_length(prompt, add_special_tokens=True)
        if prompt_tokens <= input_limit:
            break
        if not used_ids:
            raise ValueError("Prompt instructions alone exceed the available input context.")
        excess = prompt_tokens - input_limit
        used_ids = used_ids[:max(0, len(used_ids) - excess - 4)]

    return prompt, prompt_tokens, len(used_ids) < len(article_ids)

def prepare_training_split(frame: pd.DataFrame, split_name: str) -> pd.DataFrame:
    rows = []
    for _, row in frame.iterrows():
        target = build_training_target(row["summary"], row["aspects"])
        target_with_eos = target.strip() + tokenizer.eos_token
        target_tokens = token_length(target_with_eos, add_special_tokens=False)

        if target_tokens > MAX_TARGET_TOKENS:
            raise ValueError(
                f"{split_name} article {row['article_id']} target has {target_tokens} tokens; "
                "increase MAX_TARGET_TOKENS so the complete target is preserved."
            )

        prompt, prompt_tokens, article_was_truncated = fit_prompt_to_context(row["article"], target_tokens)
        prepared = row.to_dict()
        prepared["prompt"] = prompt
        prepared["training_target"] = target
        prepared["prompt_token_length"] = prompt_tokens
        prepared["target_token_length"] = target_tokens
        prepared["total_training_tokens"] = prompt_tokens + target_tokens
        prepared["article_was_truncated"] = article_was_truncated
        rows.append(prepared)

    result = pd.DataFrame(rows)
    print(f"{split_name}: {len(result)} records, truncated articles: {int(result['article_was_truncated'].sum())}")
    print(
        f"{split_name} total tokens: "
        f"median={result['total_training_tokens'].median():.0f}, "
        f"max={result['total_training_tokens'].max():.0f}"
    )
    return result

train_df = prepare_training_split(train_df, "Training")
val_df = prepare_training_split(val_df, "Validation")

test_df = test_df.copy()
test_df["training_target"] = test_df.apply(
    lambda row: build_training_target(row["summary"], row["aspects"]),
    axis=1,
)

print("\nExample prompt:\n")
print(train_df.loc[0, "prompt"][:1200])
print("\n--- TARGET ---\n")
print(train_df.loc[0, "training_target"])

tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/651 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

Training: 400 records, truncated articles: 0
Training total tokens: median=869, max=1191
Validation: 50 records, truncated articles: 0
Validation total tokens: median=858, max=1047

Example prompt:

Simplify the news article using only facts stated in the article.
Use simpler vocabulary, grammar, and sentence structures while preserving the meaning.
Do not guess information.

Output exactly this format:

Summary:
<2 to 4 concise simplified sentences>

Aspects:
Event: <main event>
People: <relevant named people only, or Not stated>
Organization: <relevant named organizations only, or Not stated>
Date & Time: <absolute dates/times only, or Not stated>
Location: <relevant locations, or Not stated>
Causes: <cause of the event, or Not stated; use "Believed cause:" if uncertain>
Effects: <results or consequences of the event, or Not stated>
Countermeasures: <actions taken in response to the event, or Not stated>

Use every label exactly once and in the same order.
Do not repeat the same info

## Create training datasets

In [6]:
class NewsSimplificationDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, tokenizer):
        self.prompts = frame["prompt"].tolist()
        self.targets = frame["training_target"].tolist()
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.prompts)

    def __getitem__(self, index):
        prompt = str(self.prompts[index])
        target = str(self.targets[index]).strip() + self.tokenizer.eos_token

        prompt_ids = self.tokenizer(
            prompt,
            add_special_tokens=True,
            truncation=False,
        )["input_ids"]

        target_ids = self.tokenizer(
            target,
            add_special_tokens=False,
            truncation=False,
        )["input_ids"]

        if len(prompt_ids) + len(target_ids) > MAX_CONTEXT:
            raise ValueError(f"Training pair {index} exceeds {MAX_CONTEXT} tokens.")

        input_ids = prompt_ids + target_ids
        labels = [-100] * len(prompt_ids) + target_ids

        return {
            "input_ids": input_ids,
            "attention_mask": [1] * len(input_ids),
            "labels": labels,
        }

def collate_fn(batch, pad_token_id: int):
    max_len = max(len(item["input_ids"]) for item in batch)
    input_ids, attention_masks, labels = [], [], []

    for item in batch:
        pad_length = max_len - len(item["input_ids"])
        input_ids.append(item["input_ids"] + [pad_token_id] * pad_length)
        attention_masks.append(item["attention_mask"] + [0] * pad_length)
        labels.append(item["labels"] + [-100] * pad_length)

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_masks, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
    }

train_dataset = NewsSimplificationDataset(train_df, tokenizer)
val_dataset = NewsSimplificationDataset(val_df, tokenizer)
safe_collator = partial(collate_fn, pad_token_id=tokenizer.pad_token_id)

print("Training examples:", len(train_dataset))
print("Validation examples:", len(val_dataset))

Training examples: 400
Validation examples: 50


## Load model with 4-bit QLoRA

In [7]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map={"": 0},
    torch_dtype=COMPUTE_DTYPE,
)

print("Model context limit:", model.config.max_position_embeddings)
if MAX_CONTEXT > model.config.max_position_embeddings:
    raise ValueError("MAX_CONTEXT exceeds the model context limit.")

model.config.use_cache = False
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules="all-linear",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/251M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/251M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Model context limit: 2048
trainable params: 2,654,208 || all params: 127,893,504 || trainable%: 2.0753


## Fine-tune

In [8]:
training_args = TrainingArguments(
    output_dir=os.path.join(FINETUNED_MODEL_DIR, "checkpoints"),
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    max_grad_norm=MAX_GRAD_NORM,
    optim="paged_adamw_8bit",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=not USE_BF16,
    bf16=USE_BF16,
    gradient_checkpointing=True,
    remove_unused_columns=False,
    report_to=[],
    seed=SEED,
    data_seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=safe_collator,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE
        )
    ],
)

train_result = trainer.train()
print(train_result)
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best eval loss:", trainer.state.best_metric)

Step,Training Loss,Validation Loss
25,1.708600,1.510195
50,1.322800,1.274366
75,1.182200,1.183305
100,1.049500,1.133905
125,1.149500,1.109278
150,1.069900,1.095599
175,1.070600,1.080394
200,0.990000,1.074603
225,1.006100,1.070279
250,1.006600,1.066496


TrainOutput(global_step=475, training_loss=1.0722142560858476, metrics={'train_runtime': 663.4408, 'train_samples_per_second': 6.029, 'train_steps_per_second': 0.754, 'total_flos': 1710034878034944.0, 'train_loss': 1.0722142560858476, 'epoch': 9.5})
Best checkpoint: /kaggle/working/models/opt-125m-news-simplifier/checkpoints/checkpoint-375
Best eval loss: 1.0560083389282227


## Save model and training log

In [9]:
trainer.save_model(FINETUNED_MODEL_DIR)
tokenizer.save_pretrained(FINETUNED_MODEL_DIR)

training_log_path = os.path.join(RESULTS_DIR, f"{RESULT_PREFIX}_training_log.csv")
pd.DataFrame(trainer.state.log_history).to_csv(training_log_path, index=False)

metadata = {
    "base_model": MODEL_NAME,
    "saved_model_dir": FINETUNED_MODEL_DIR,
    "training_method": "4-bit QLoRA / PEFT",
    "max_context": MAX_CONTEXT,
    "max_target_tokens": MAX_TARGET_TOKENS,
    "train_records_used": len(train_df),
    "validation_records_used": len(val_df),
    "training_articles_truncated": int(train_df["article_was_truncated"].sum()),
    "validation_articles_truncated": int(val_df["article_was_truncated"].sum()),
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "best_eval_loss": trainer.state.best_metric,
    "seed": SEED,
    "lora": {
        "r": LORA_R,
        "alpha": LORA_ALPHA,
        "dropout": LORA_DROPOUT,
        "target_modules": "all-linear",
    },
    "training": {
        "max_epochs": NUM_TRAIN_EPOCHS,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "eval_steps": EVAL_STEPS,
        "save_steps": SAVE_STEPS,
        "batch_size": TRAIN_BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "effective_batch_size": TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "warmup_ratio": WARMUP_RATIO,
        "max_grad_norm": MAX_GRAD_NORM,
        "optimizer": "paged_adamw_8bit",
        "lr_scheduler": "cosine",
        "compute_dtype": str(COMPUTE_DTYPE),
    },
    "software": {
        "python": platform.python_version(),
        "pytorch": torch.__version__,
        "transformers": transformers.__version__,
        "peft": peft.__version__,
        "accelerate": accelerate.__version__,
        "bitsandbytes": bnb.__version__,
    },
}

with open(
    os.path.join(FINETUNED_MODEL_DIR, "training_metadata.json"),
    "w",
    encoding="utf-8",
) as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print(f"Saved PEFT adapter and tokenizer to '{FINETUNED_MODEL_DIR}'")
print(f"Saved training log to '{training_log_path}'")

Saved PEFT adapter and tokenizer to '/kaggle/working/models/opt-125m-news-simplifier'
Saved training log to '/kaggle/working/results/opt125m_training_log.csv'


## Single-pass test generation

In [10]:
finetuned_model = trainer.model
finetuned_model.eval()
finetuned_model.config.use_cache = True

if hasattr(finetuned_model, "gradient_checkpointing_disable"):
    finetuned_model.gradient_checkpointing_disable()

torch.cuda.empty_cache()

OUTPUT_LABELS = (
    "Event",
    "People",
    "Organization",
    "Date & Time",
    "Causes",
    "Location",
    "Effects",
    "Countermeasures",
)

ASPECT_FIELD_MAP = {
    "Event": "event",
    "People": "people",
    "Organization": "organization",
    "Date & Time": "date_time",
    "Location": "location",
    "Causes": "causes",
    "Effects": "effects",
    "Countermeasures": "countermeasures",
}

GENERATION_DEVICE = next(
    parameter.device
    for parameter in finetuned_model.parameters()
    if parameter.device.type != "meta"
)

print("Generation device:", GENERATION_DEVICE)

def generate_article_output(article: str):
    prompt, input_token_length, article_was_truncated = fit_prompt_to_context(
        article,
        MAX_NEW_TOKENS,
    )

    inputs = tokenizer(
        prompt,
        add_special_tokens=True,
        truncation=False,
        return_tensors="pt",
    )

    inputs = {
        key: value.to(GENERATION_DEVICE)
        for key, value in inputs.items()
    }

    input_length = inputs["input_ids"].shape[1]

    start_time = time.perf_counter()

    with torch.inference_mode():
        output_ids = finetuned_model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    elapsed = time.perf_counter() - start_time
    generated_ids = output_ids[0][input_length:]
    raw_output = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    ).strip()

    return {
        "raw_output": raw_output,
        "generation_time_sec": elapsed,
        "input_token_length": input_token_length,
        "article_was_truncated": article_was_truncated,
    }

def extract_summary(raw_text: str) -> str:
    match = re.search(
        r"(?is)^\s*Summary\s*:\s*(.*?)(?=\n\s*Aspects\s*:|\Z)",
        str(raw_text),
    )
    return re.sub(r"\s+", " ", match.group(1)).strip() if match else ""

def extract_aspects(raw_text: str) -> dict:
    text = str(raw_text)
    extracted = {}

    for label in OUTPUT_LABELS:
        match = re.search(
            rf"(?im)^\s*{re.escape(label)}\s*:\s*(.*?)\s*$",
            text,
        )
        extracted[ASPECT_FIELD_MAP[label]] = match.group(1).strip() if match else ""

    return extracted

def has_complete_structure(raw_text: str) -> bool:
    text = str(raw_text)
    ordered_labels = ["Summary", "Aspects", *OUTPUT_LABELS]
    positions = []

    for label in ordered_labels:
        found = list(
            re.finditer(
                rf"(?im)^\s*{re.escape(label)}\s*:",
                text,
            )
        )

        if len(found) != 1:
            return False

        positions.append(found[0].start())

    return positions == sorted(positions)

def exact_label_completion_rate(raw_text: str) -> float:
    text = str(raw_text)
    labels = ["Summary", "Aspects", *OUTPUT_LABELS]

    present = sum(
        bool(
            re.search(
                rf"(?im)^\s*{re.escape(label)}\s*:",
                text,
            )
        )
        for label in labels
    )

    return present / len(labels)

Generation device: cuda:0


In [11]:
generated_rows = []

for position, (_, row) in enumerate(test_df.iterrows(), start=1):
    generated_rows.append(generate_article_output(row["article"]))

    if position % 10 == 0 or position == len(test_df):
        print(f"Generated {position}/{len(test_df)} responses")

generated_df = pd.DataFrame(generated_rows)
test_df = pd.concat(
    [test_df.reset_index(drop=True), generated_df],
    axis=1,
)

test_df["generated_summary"] = test_df["raw_output"].apply(extract_summary)
test_df["generated_aspects"] = test_df["raw_output"].apply(extract_aspects)
test_df["structured_output_complete"] = test_df["raw_output"].apply(has_complete_structure)
test_df["exact_label_completion_rate"] = test_df["raw_output"].apply(exact_label_completion_rate)

test_df[
    [
        "summary",
        "generated_summary",
        "raw_output",
        "structured_output_complete",
        "exact_label_completion_rate",
    ]
].head()

Generated 10/50 responses
Generated 20/50 responses
Generated 30/50 responses
Generated 40/50 responses
Generated 50/50 responses


,summary,generated_summary,raw_output,structured_output_complete,exact_label_completion_rate
0,Police discovered the room when called to a di...,"Bradley Bleimeyer, 24, and his 16-year-old ste...","Summary:\nBradley Bleimeyer, 24, and his 16-ye...",False,1.0
1,Mario Balotelli is back in the Italy squad aft...,Mario Balotelli is back in favour of Italy's f...,Summary:\nMario Balotelli is back in favour of...,False,1.0
2,Researchers admit they are baffled by the resu...,Scientists have discovered the shape of the Su...,Summary:\nScientists have discovered the shape...,False,1.0
3,"Colton Clegg, three, died after being kicked b...",Colton Lee Clegg was helping his father and un...,Summary:\nColton Lee Clegg was helping his fat...,False,1.0
4,"NEW: The fire has burned nearly 1,900 acres an...","Mariposa, California, is threatened by a Carst...","Summary:\nMariposa, California, is threatened ...",False,1.0


## Evaluate

In [13]:
scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True,
)

def compute_rouge(reference: str, hypothesis: str):
    scores = scorer.score(str(reference), str(hypothesis))
    return {
        "rouge1": scores["rouge1"].fmeasure,
        "rouge2": scores["rouge2"].fmeasure,
        "rougeL": scores["rougeL"].fmeasure,
    }

def rouge_l(reference: str, hypothesis: str) -> float:
    return scorer.score(
        str(reference),
        str(hypothesis),
    )["rougeL"].fmeasure

def word_count(text: str) -> int:
    return max(1, len(str(text).split()))

def aspect_macro_rouge_l(
    reference_aspects: dict,
    generated_aspects: dict,
) -> float:
    return float(
        np.mean(
            [
                rouge_l(
                    reference_aspects[key],
                    generated_aspects.get(key, ""),
                )
                for key in ASPECT_KEYS
            ]
        )
    )

rouge1_scores, rouge2_scores, rougeL_scores = [], [], []
full_target_rougeL_scores, aspect_macro_scores = [], []
flesch_scores, compression_ratios = [], []

for _, row in test_df.iterrows():
    summary_rouge = compute_rouge(
        row["summary"],
        row["generated_summary"],
    )

    rouge1_scores.append(summary_rouge["rouge1"])
    rouge2_scores.append(summary_rouge["rouge2"])
    rougeL_scores.append(summary_rouge["rougeL"])

    full_target_rougeL_scores.append(
        rouge_l(
            row["training_target"],
            row["raw_output"],
        )
    )

    aspect_macro_scores.append(
        aspect_macro_rouge_l(
            row["aspects"],
            row["generated_aspects"],
        )
    )

    flesch_scores.append(
        textstat.flesch_reading_ease(row["generated_summary"])
        if row["generated_summary"]
        else 0.0
    )

    compression_ratios.append(
        word_count(row["generated_summary"])
        / word_count(row["article"])
    )

test_df["rouge1"] = rouge1_scores
test_df["rouge2"] = rouge2_scores
test_df["rougeL"] = rougeL_scores
test_df["full_target_rougeL"] = full_target_rougeL_scores
test_df["aspect_macro_rougeL"] = aspect_macro_scores
test_df["flesch_reading_ease"] = flesch_scores
test_df["compression_ratio"] = compression_ratios

## Save evaluation results

In [14]:
summary_metrics = pd.DataFrame(
    {
        "metric": [
            "ROUGE-1",
            "ROUGE-2",
            "ROUGE-L",
            "Full Target ROUGE-L",
            "Aspect Macro ROUGE-L",
            "Flesch Reading Ease",
            "Compression Ratio",
            "Generation Time (sec)",
            "Structured Output Completion Rate",
            "Exact Label Completion Rate",
            "Test Article Truncation Rate",
        ],
        "mean": [
            test_df["rouge1"].mean(),
            test_df["rouge2"].mean(),
            test_df["rougeL"].mean(),
            test_df["full_target_rougeL"].mean(),
            test_df["aspect_macro_rougeL"].mean(),
            test_df["flesch_reading_ease"].mean(),
            test_df["compression_ratio"].mean(),
            test_df["generation_time_sec"].mean(),
            test_df["structured_output_complete"].mean(),
            test_df["exact_label_completion_rate"].mean(),
            test_df["article_was_truncated"].mean(),
        ],
        "std": [
            test_df["rouge1"].std(),
            test_df["rouge2"].std(),
            test_df["rougeL"].std(),
            test_df["full_target_rougeL"].std(),
            test_df["aspect_macro_rougeL"].std(),
            test_df["flesch_reading_ease"].std(),
            test_df["compression_ratio"].std(),
            test_df["generation_time_sec"].std(),
            test_df["structured_output_complete"].std(),
            test_df["exact_label_completion_rate"].std(),
            test_df["article_was_truncated"].std(),
        ],
    }
)

print(summary_metrics.to_string(index=False))

per_example_path = os.path.join(
    RESULTS_DIR,
    f"{RESULT_PREFIX}_test_results.csv",
)

summary_path = os.path.join(
    RESULTS_DIR,
    f"{RESULT_PREFIX}_evaluation_summary.csv",
)

save_df = test_df.copy()

save_df["aspects"] = save_df["aspects"].apply(
    lambda value: json.dumps(value, ensure_ascii=False)
)

save_df["generated_aspects"] = save_df["generated_aspects"].apply(
    lambda value: json.dumps(value, ensure_ascii=False)
)

save_df.drop(
    columns=["training_target"],
    errors="ignore",
).to_csv(per_example_path, index=False)

summary_metrics.to_csv(summary_path, index=False)

print(f"Saved test results to '{per_example_path}'")
print(f"Saved evaluation summary to '{summary_path}'")

                           metric      mean       std
                          ROUGE-1  0.327681  0.122756
                          ROUGE-2  0.121661  0.091897
                          ROUGE-L  0.238230  0.100315
              Full Target ROUGE-L  0.379038  0.087035
             Aspect Macro ROUGE-L  0.552644  0.164447
              Flesch Reading Ease 56.977848 17.215705
                Compression Ratio  0.155374  0.108762
            Generation Time (sec)  4.373883  1.054473
Structured Output Completion Rate  0.000000  0.000000
      Exact Label Completion Rate  1.000000  0.000000
     Test Article Truncation Rate  0.000000  0.000000
Saved test results to '/kaggle/working/results/opt125m_test_results.csv'
Saved evaluation summary to '/kaggle/working/results/opt125m_evaluation_summary.csv'


## CHANGE NAME TO RESPECTIVE MODEL

In [15]:
!zip -r opt-125m.zip /kaggle/working/models/opt-125m-news-simplifier


  adding: kaggle/working/models/opt-125m-news-simplifier/ (stored 0%)
  adding: kaggle/working/models/opt-125m-news-simplifier/adapter_model.safetensors (deflated 8%)
  adding: kaggle/working/models/opt-125m-news-simplifier/training_args.bin (deflated 53%)
  adding: kaggle/working/models/opt-125m-news-simplifier/checkpoints/ (stored 0%)
  adding: kaggle/working/models/opt-125m-news-simplifier/checkpoints/checkpoint-450/ (stored 0%)
  adding: kaggle/working/models/opt-125m-news-simplifier/checkpoints/checkpoint-450/adapter_model.safetensors (deflated 8%)
  adding: kaggle/working/models/opt-125m-news-simplifier/checkpoints/checkpoint-450/scaler.pt (deflated 64%)
  adding: kaggle/working/models/opt-125m-news-simplifier/checkpoints/checkpoint-450/training_args.bin (deflated 53%)
  adding: kaggle/working/models/opt-125m-news-simplifier/checkpoints/checkpoint-450/adapter_config.json (deflated 59%)
  adding: kaggle/working/models/opt-125m-news-simplifier/checkpoints/checkpoint-450/rng_state.p

In [16]:
!zip -r opt-125m-results.zip /kaggle/working/results

  adding: kaggle/working/results/ (stored 0%)
  adding: kaggle/working/results/opt125m_test_results.csv (deflated 73%)
  adding: kaggle/working/results/opt125m_training_log.csv (deflated 57%)
  adding: kaggle/working/results/opt125m_evaluation_summary.csv (deflated 37%)
